# 01.2 — Control flow

`if / for / while` in Python, with the MATLAB equivalent next to each. This notebook also covers the "Pythonic" patterns that MATLAB doesn't have — list comprehensions, `enumerate`, `zip`, `for/else` — because production code uses them liberally.

**Prerequisite:** [01.1 syntax basics](01.1_syntax_basics.ipynb).

## Section 1 — What MATLAB does

```matlab
% if/elseif/else/end
if x > 10
    fprintf('big\n');
elseif x > 0
    fprintf('small\n');
else
    fprintf('non-positive\n');
end

% for loop over a vector
for k = 1:10
    fprintf('%d ', k);
end

% while
n = 0;
while n < 5
    n = n + 1;
end

% break / continue work the same in both languages.
```

Python uses the same keywords with different syntax: **no `end`** (indentation marks the block end), **colons after the condition**, and **half-open ranges** (`range(1, 10)` is `1, 2, ..., 9` — not `10`).

## Section 2 — The Python concepts you need

### 2.1 — `if` / `elif` / `else`

Three things to remember: colons, `elif` (not `elseif`), indentation.

In [ ]:
x = 7

if x > 10:
    print("big")
elif x > 0:
    print("small")
else:
    print("non-positive")

In [ ]:
# Multiple conditions chain with `and` / `or` (not && / ||)
y = 5
if 0 < y < 10:                      # Python supports chained comparisons
    print("in range")

if y > 0 and y % 2 != 0:
    print("positive odd")

**Chained comparisons** are a Pythonism MATLAB doesn't have. `0 < y < 10` is shorthand for `(0 < y) and (y < 10)`. Very common in production code; useful for bounds checks.

### 2.2 — `for` loops over collections

In MATLAB you iterate by index: `for k = 1:length(arr)`, then use `arr(k)`. In Python you iterate directly over the collection's elements.

In [ ]:
fruits = ["apple", "banana", "cherry"]
for fruit in fruits:                 # iterate over elements, NOT indices
    print(fruit)

In [ ]:
# When you need BOTH the index and the value: enumerate
for idx, fruit in enumerate(fruits):
    print(f"{idx}: {fruit}")

# Or starting the count from 1 (closer to MATLAB):
print()
for idx, fruit in enumerate(fruits, start=1):
    print(f"{idx}: {fruit}")

In [ ]:
# Iterating over a range of integers — MATLAB's for k = 1:N
for k in range(5):       # 0, 1, 2, 3, 4
    print(k, end=" ")
print()

for k in range(1, 6):    # 1, 2, 3, 4, 5
    print(k, end=" ")
print()

for k in range(0, 10, 2):    # step
    print(k, end=" ")
print()

**The `range()` mental model:** `range(start, stop, step)` is a lazy generator (it doesn't allocate the full list). `range(1, 5)` covers `1, 2, 3, 4` — `stop` is **exclusive**. Use `range(1, n + 1)` if you really want a 1-indexed loop.

### 2.3 — Iterating over multiple sequences at once: `zip`

The MATLAB pattern `for k=1:length(a); use(a(k), b(k)); end` becomes a `zip` in Python.

In [ ]:
names = ["GRU", "LSTM", "Feedforward"]
sizes = [1000, 500, 250]

for name, size in zip(names, sizes):
    print(f"{name}: hidden={size}")

In [ ]:
# Three or more sequences — zip handles them all
losses = ["MSE", "MAE", "None"]
weights = [100, 1, 0]
flags = [True, True, False]

for loss, w, flag in zip(losses, weights, flags):
    if flag:
        print(f"{loss} weight={w}")

### 2.4 — `while` loops

Identical mechanics to MATLAB. Used less often than `for` because Python's iteration tools are richer.

In [ ]:
n = 0
while n < 5:
    n += 1     # same as `n = n + 1`. Python has NO `++` operator.
print(n)

### 2.5 — `break` and `continue`

Same behavior as MATLAB. `break` exits the innermost loop; `continue` skips to the next iteration.

In [ ]:
for n in range(10):
    if n == 3:
        continue       # skip 3
    if n == 7:
        break          # stop entirely
    print(n, end=" ")
print()

### 2.6 — List comprehensions (the Pythonic idiom MATLAB lacks)

A list comprehension is a one-line `for` loop that builds a list. MATLAB does this with `arrayfun(@(x) f(x), arr)` — but the Python syntax is much more readable.

In [ ]:
# Equivalent of: result = []; for x in arr: result.append(x * 2)
squares = [x * x for x in range(10)]
print(squares)

In [ ]:
# With a filter clause
even_squares = [x * x for x in range(10) if x % 2 == 0]
print(even_squares)

In [ ]:
# Nested comprehension — equivalent of two nested for loops
pairs = [(i, j) for i in range(3) for j in range(3) if i != j]
print(pairs)

In [ ]:
# Dict comprehension — build a dict the same way
hidden_by_model = {name: size for name, size in zip(names, sizes)}
print(hidden_by_model)

**When to use a comprehension vs a plain for loop:**

- Use a comprehension when the loop body is a single expression that builds a collection.
- Use a plain `for` loop when the body has side effects, multiple statements, or `break`/`continue` logic.

Comprehensions are widely used in `src/neural_data_decoding/` — search for `[`...`for`...`in`...`]` patterns to see how often.

### 2.7 — `for / else` — the obscure-but-useful Python feature

Python's `for` loop can have an `else` clause that runs **iff the loop completed without `break`**. Useful for "search and report not found" patterns.

In [ ]:
haystack = [1, 4, 7, 9]
target = 5

for x in haystack:
    if x == target:
        print(f"found {target}")
        break
else:
    print(f"{target} not in haystack")

**`while / else`** works the same way. Rare in our codebase but you'll see it in idiomatic Python.

## Section 3 — The `neural_data_decoding` implementation

Search the codebase for a control-flow pattern you saw in MATLAB and find its Python analog.

In [ ]:
# Production example: iterating over sweep entries
from neural_data_decoding.sweeps import SWEEP_ENTRIES, iter_by_choice

# Tally entries by MATLAB SLURMChoice using the iter_by_choice helper.
for choice, entries in iter_by_choice():
    print(f"SC{choice}: {len(entries)} entries")

In [ ]:
# A list comprehension extracting all entries with a partial-support note
entries_with_notes = [e for e in SWEEP_ENTRIES if e.notes]
print(f"{len(entries_with_notes)} entries have caveats")
for e in entries_with_notes[:3]:
    print(f"  ({e.matlab_choice}, {e.matlab_idx}) {e.description}: {e.notes[0][:60]}...")

## Section 4 — Hands-on exercises

### Exercise 4.1 — translate a MATLAB loop

Port:

```matlab
total = 0;
for k = 1:10
    if mod(k, 2) == 0
        total = total + k;
    end
end
fprintf('total = %d\n', total);
```

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
total = 0
for k in range(1, 11):
    if k % 2 == 0:
        total += k
print(f"total = {total}")

### Exercise 4.2 — rewrite as a comprehension

Take Exercise 4.1 and rewrite the entire loop as a one-line `sum(...)` with a comprehension.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
total = sum(k for k in range(1, 11) if k % 2 == 0)
print(f"total = {total}")
# Note: this is a GENERATOR expression (no [ ]) passed to sum().
# Same as sum([k for k in range(1, 11) if k % 2 == 0]) but doesn't allocate the list.

## Section 5 — Common errors

### `SyntaxError: expected ':'`

You wrote `if x > 0` without the colon. Python requires it on every block-opening line: `if`, `elif`, `else`, `for`, `while`, `def`, `class`, `try`, `except`, `with`.

### `for k = 1:10` doesn't work

That's MATLAB syntax. Use `for k in range(1, 11):` (note the colon and the half-open `11`).

### `n++` doesn't work

Python has no increment/decrement operators. Use `n += 1` (which is shorthand for `n = n + 1`).

### `if x == True:` linter warning

Prefer `if x:` for booleans. The explicit `== True` is redundant and frowned upon. Use `is True` only when you genuinely want to check identity (rare).

### Off-by-one when porting

MATLAB `1:N` covers `1..N` inclusive. Python `range(1, N)` covers `1..N-1`. To cover `1..N` inclusive, use `range(1, N + 1)`.

### Mutating the list you're iterating

```python
nums = [1, 2, 3, 4]
for n in nums:
    if n % 2 == 0:
        nums.remove(n)    # WRONG — modifies the list during iteration
```

Build a new list instead: `nums = [n for n in nums if n % 2 != 0]`.

## Section 6 — Further reading

- [Python data model — for loops](https://docs.python.org/3/reference/compound_stmts.html#for) — the canonical reference, including the `for / else` semantics.
- [PEP 202 — list comprehensions](https://peps.python.org/pep-0202/). The original proposal; explains the design.
- [Python's `itertools` module](https://docs.python.org/3/library/itertools.html). `chain`, `groupby`, `accumulate`, `takewhile`, etc. — the Pythonic version of MATLAB's array-processing functions.

Next up: **[01.3 functions and lambdas](01.3_functions_and_lambdas.ipynb)**.